# Enterprise RAG — Hands-On, Part 1 of 11: The corpus and its permissions

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [1]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

project root : d:\INTERVIEW PREPARATION\DevRev_Preparation\enterprise_rag_platform
corpus       : data\corpus
api key      : found
embed model  : text-embedding-3-small
chat model   : gpt-4o-mini


---
# Part 1 - The corpus and its permissions

Content and permissions are two separate feeds, joined by `doc_id`:

- **Content** - `data/corpus/*.md`. Frontmatter carries only `doc_id` and `title`; the body is the
  document text. No access-control field lives here.
- **Permissions** - `data/acl_manifest.json`. One JSON record per `doc_id`: `sensitivity`,
  `allowed_groups`, `region`, `source`, `need_to_know`, `contains_pii`, `valid_from`/`valid_until`.
  This is the stand-in for whatever system actually owns entitlements in production - an admin
  console, an HR/entitlements system, a Confluence-space-permissions export.

`load_corpus()` joins the two by `doc_id`. A content file with no matching manifest record is
refused outright - there is no "default to internal" fallback. Getting that join wrong is the number
one cause of enterprise RAG leaks, which is why it lives in its own, boringly explicit function.

**Where this ends up:** the join happens once, at ingest time (Part 4). From there the resulting
`ResourceAttributes` are written to *two* places with different jobs - a denormalised copy on each
chunk in the vector index (a cache, used only to make retrieval cheap) and a row in a separate local
ACL catalog (SQLite), which is the *authoritative* copy the post-retrieval policy check actually
reads.

In [2]:
from enterprise_rag.ingest.loader import load_corpus

docs = load_corpus()
print(f"{len(docs)} documents\n")
print(f"{'doc_id':<16}{'source':<12}{'sensitivity':<14}{'region':<8}{'allowed_groups'}")
print("-" * 92)
for d in sorted(docs, key=lambda x: (x.attrs.source, x.attrs.doc_id)):
    a = d.attrs
    extra = ""
    if a.need_to_know:
        extra += f"  need-to-know={a.need_to_know}"
    if a.valid_from:
        extra += f"  embargoed until {a.valid_from}"
    if a.contains_pii:
        extra += "  [PII]"
    print(f"{a.doc_id:<16}{a.source:<12}{a.sensitivity:<14}{a.region:<8}"
          f"{','.join(a.allowed_groups)}{extra}")

22 documents

doc_id          source      sensitivity   region  allowed_groups
--------------------------------------------------------------------------------------------
SA-2026-05      advisory    restricted    GLOBAL  security,engineering  need-to-know=['vuln-response']  embargoed until 2026-05-15
SA-2026-07      advisory    restricted    GLOBAL  security  need-to-know=['vuln-response']  embargoed until 2026-09-01
CT-KST-003      contract    confidential  GLOBAL  sales,legal,account-management
CT-NGR-002      contract    confidential  US      sales,legal,account-management
CT-VTX-001      contract    confidential  EU      sales,legal,account-management
HC-001          helpcenter  public        GLOBAL  public
HC-002          helpcenter  public        GLOBAL  public
HC-003          helpcenter  public        GLOBAL  public
HC-004          helpcenter  public        GLOBAL  public
PM-2025-11-03   postmortem  confidential  GLOBAL  engineering,sre
PM-2026-01-22   postmortem  confidential 

In [3]:
# The manifest is the source of truth for access control; frontmatter no longer carries it.
print((SETTINGS.corpus_dir / "PM-2026-03-14.md").read_text(encoding="utf-8"))

---
doc_id: PM-2026-03-14
title: Post-mortem: EU Ingest Degradation, 14 March 2026
---

# Post-mortem - EU Ingest Degradation, 14 March 2026

**Status:** Final   **Severity:** SEV1   **Duration:** 08:47 - 10:34 UTC (107 minutes)
**Customer impact:** 412 EU workspaces saw sustained MRD-5031. An estimated 1.8 billion data
points were rejected. Customers on agent versions below 3.2 lost that data permanently.

## Root cause

A single workspace (ws_lmb_eu_077) deployed an instrumentation change that added a unique request
ID as a metric tag. This raised its active series count from 90,000 to 14.2 million in under
twenty minutes.

The cardinality explosion produced a flood of tiny segments. The compaction queue saturated at
roughly 08:47. Because compaction and ingest share the same storage-tier write path, backpressure
propagated to every workspace in the EU region, not just the offending one.

**The core design flaw: there is no per-workspace isolation on the compaction path.** One tenant

One document's content frontmatter, next to its permissions record - two files, one `doc_id`.

In [4]:
import json

manifest = json.loads(SETTINGS.acl_manifest_file.read_text(encoding="utf-8"))
record = next(r for r in manifest["documents"] if r["doc_id"] == "PM-2026-03-14")
print(json.dumps(record, indent=2))

{
  "doc_id": "PM-2026-03-14",
  "source": "postmortem",
  "sensitivity": "confidential",
  "allowed_groups": [
    "engineering",
    "support-tier3",
    "sre"
  ],
  "region": "EU",
  "product": "ingest",
  "owner": "ingest-team",
  "contains_pii": false,
  "need_to_know": [],
  "valid_from": null,
  "valid_until": null
}


### The people

Eight personas, each chosen to exercise a *different* policy rule. Note the last one: a principal from
another tenant holding **every** group and the highest clearance. It is the negative control - it must
never see anything at all.

In [5]:
from enterprise_rag.identity import list_principals

header = (
    f"{'user_id':<22} {'role':<24} {'clearance':<13} {'region':<6} "
    f"{'pii':^3} {'ext':^3}  groups"
)
print(header)
print("-" * len(header))

for p in list_principals():
    print(
        f"{p.user_id:<22} {p.role:<24} {p.clearance:<13} {p.region:<6} "
        f"{'Y' if p.can_view_pii else '.':^3} {'Y' if p.is_external else '.':^3}  "
        f"{', '.join(p.groups)}"
    )
    if p.projects:
        print(f"{'':<22} {'':<24} {'':<13} {'':<6} {'':^3} {'':^3}  "
              f"projects: {', '.join(p.projects)}")

print("\npii/ext: Y = yes, . = no")

user_id                role                     clearance     region pii ext  groups
------------------------------------------------------------------------------------
u_lena_t1              Tier 1 Support Agent     internal      EU      Y   .   support-tier1
u_marco_t3             Tier 3 Escalation Engineer confidential  EU      Y   .   support-tier3, engineering
u_sofia_am             Enterprise Account Manager confidential  EU      Y   .   sales, account-management
u_ravi_sec             Security Engineer        restricted    GLOBAL  Y   .   security, engineering
                                                                              projects: vuln-response
u_erin_secmgr          Security Manager (no vuln-response compartment) restricted    GLOBAL  Y   .   security
u_tom_contractor       Tier 1 Support Agent (external contractor) internal      US      .   Y   support-tier1
u_dana_ext             External Solutions Consultant (high clearance) confidential  GLOBAL  .   Y   sal

---

**Next ▶:** [2. The policy engine](part02-policy-engine.ipynb)
